# Project II: Janus

## 1. Environment Setup and Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Tensorflow / keras for CNN and ArcFace
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#  YOLOv8 for face detection
!pip install ultralytics
from ultralytics import YOLO

#  InsightFace (ArcFace implementation)
!pip install InsightFace
!pip install onnxruntime
from insightface.app import FaceAnalysis

#  function to extract cartel member name from image path
def get_cartel_member_name_from_path(img_path):

    path_parts = img_path.split(os.sep)
    try:
        sinaloa_index = path_parts.index('Sinaloa')
        if sinaloa_index + 1 < len(path_parts):
            return path_parts[sinaloa_index + 1]
    except:
        pass # 'sinaloa' not in path

    if 'SinaloaTest/carteltest3.png' in img_path: #  specific case from previous execution
        return 'Unknown member'

    return os.path.basename(os.path.dirname(img_path))#  fallback to parent directory name

print(f'Tensowflow Version: {tf.__version__}')

## 2. Dataset Loading and preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = '/content/drive/MyDrive/Project_Janus/Sinaloa'

# image data generators for CNN
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

class_names = list(train_generator.class_indices.keys())
print('target cartel targets:', class_names)

## 3. Viewing the images from the directory

In [ ]:
import os

#  we are writing a for loop to read all the items in our google drive
for item_name in os.listdir(data_dir):
    print(item_name)
    #  use item_name (which is in the class name) to construct the path to the subdirectory
    for sub_item_name in os.listdir(os.path.join(data_dir, item_name)):
        print(f'   {sub_item_name}')

## 4. CNNs for Face Detection

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

#  1. aggressive Data Augmentation to prevent overfitting on aerial angles
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2)
])

#  adapting standard CNN architecture for face classification
cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(class_names), activation='softmax') #  traditional softmax
])

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=
            ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

#  train the CNN model
history_cnn = cnn_model.fit(
    train_generator,
    epochs=25,
    validation_data=val_generator
)

## 4.1 Performance metrics for CNN model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

#  1. Plot training history
acc = history_cnn.history['accuracy']
val_acc = history_cnn.history['val_accuracy']
loss = history_cnn.history['loss']
val_loss = history_cnn.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
# plt.show() # Removed to prevent showing two separate plots

plt.subplot(1, 2, 2) # Changed from (1, 2, 1) to (1, 2, 2)
plt.plot(loss, label='Training loss')
plt.plot(val_loss, label='Validation loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

#  2. Generate predictions for the validation set
print('\nEvaluating model performance...')
y_true = []
y_pred= []

# Iterate through val_generator instead of undefined val_ds
for images, labels in val_generator:
    # Use cnn_model for prediction
    preds = cnn_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels, axis=1)) # Convert one-hot encoded labels to class indices

#  3. Classification report (precision, recall, f1-score)
print('\nClassification report: ')
print(classification_report(y_true, y_pred, target_names=class_names))

#  4. Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('predicted jet')
plt.ylabel('True jet')
plt.title('Confusion Matrix: Russian Fighter Jets')
plt.show()

## 4.2 Confusion Matrix of the CNN model

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

#  generate predictions for the validation set
val_generator.reset()
y_pred_probs = cnn_model.predict(val_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

#  get true labels
y_true = val_generator.classes

#  3. classification report (precision, recall, f1)
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

#  4. confusion matrix (critical for diagnosing misclassifictation)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('predicted cartel member')
plt.ylabel('true cartel member')
plt.title('confusion matrix: sinaloa cartel members')
plt.show()

## 4.3 Testing the CNN model

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image

print('Please upload an image of a member of Cartel Member for detection')

uploaded = files.upload()

for fn in uploaded.keys():
    path = '/content/' + fn

    img = image.load_img(path, target_size=(IMG_HEIGHT, IMG_WIDTH))

    #  3. convert image to array and add a batch dimension (shape becomes 1 x 224 x 224 x 3)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    #  4. make predictions
    predictions = cnn_model.predict(img_array)

    #  5. extract the class with the highest probability
    confidence = np.max(predictions[0]) * 100
    predicted_class_index = np.argmax(predictions[0])
    predicted_class_name = class_names[predicted_class_index]

    #  6. dispaly the image alongside the prediction results
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Detected: {predicted_class_name}\nConfidence: {confidence:.2f}%',
              fontsize=14, fontweight='bold', color='green' if confidence > 50 else 'red')
    plt.show

## 5. YOYO v8 for Face Detection

In [ ]:
#  install required libraries
!pip install -q split-folders ultralytics

import splitfolders
from ultralytics import YOLO

#  1. Split data into train/val folders for YOLOv8 architechture
output_yolo_dir = '/content/yolo_dataset'
splitfolders.ratio(data_dir, output=output_yolo_dir, seed=42, ratio=(.8,.2), group_prefix=None)

#  2. initialize YOLOv8 classification model (nano verrsion for speed)
yolo_model = YOLO('yolov8n-cls.pt')

#  3. train Yolov8
#  YOLO automatically logs metrics, confusion matricies, and weights to the 'runs/classify/train' directory
results = yolo_model.train(
    data=output_yolo_dir,
    epochs=25,
    imgsz=224,
    batch=32,
    name='project_janus_yolov8'
)

#  4. valibate the YOLO model
metrics = yolo_model.val()
print(f'YOLOv8 Top-1 Accuracy: {metrics.top1:.4f}')

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mping
from google.colab import files
from ultralytics import YOLO

print('please upload an image of a cartel member for YOLOv8 classification:')
uploaded = files.upload()

#  1. locate the trained weights
#  yolov8 saves these automatically during the model.train() step.
#  nga_skywatch_yolov8 matches the 'name' parameter from the training code
trained_weights_path = 'run/classify/cartel_yolov8/weights/best.pt'

#  load the trained model (or fallback to the base classification model if the path is disrupted)
if os.path.exists(trained_weights_path):
    inference_model = YOLO(trained_weights_path)
else:
    print(f'Warning: Trained weights not found at {trained_weights_path}. Using the model from the memory')
    inference_model = yolo_model

for fn in uploaded.keys():
    #  2. get the path of the uploaded image
    img_path = '/content/' + fn

    #  3. run YOLO inference on the image
    results = inference_model.predict(img_path)

    #  4. extract the prediction details from the results object
    result = results[0]
    top_class_index = result.probs.top1
    confidence = result.probs.top1conf.item() * 100
    predicted_class_name = result.names[top_class_index]

    #  5. display the image alongside the YOLO prediction results
    img = mpimg.imread(img_path)
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'YOLOv8 Detected: {predicted_class_name}\nCOnfidence: {confidence:.2f}%',
              fontsize=14, fontweight='bold', color='green' if confidence > 50 else 'red')
    plt.show()


## AcrFAce for Highly Discrimitive Embeddings

In [ ]:
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))

def extract_arcface_embeddings(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f'Error: could not load image from {img_path}. Please check the path.')
        return None, None, None
    faces = app.get(img)
    if len(faces) == 0:
        print('No faces detected by AcrFace.')
        return None, None, img

    #  assuming one primary face per bounding box
    embedding = face[0].embeedding
    return embedding, faces[0].bbox, img

def test_arcface_recognition(img_path_1, img_path_2):
    #  extract embeddings for two faces
    emb1, bbox1, img1, = extract_arcface_embeddings(img_path_1)
    emb2, bbox2, img2, = extract_arcface_embeddings(img_path_2)

    if emb1 is not None and emb2 is not None:
        #  calculate cosine similarity between the highly descrimitive face embeddings
        similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

        match_status = 'MATCH' if similarity > 0.45 else 'NO MATCH'
        color = 'green' if match_status == 'MATCH' else 'red'

        #  get cartel member names
        name1 = get_cartel_member_name_from_path(img_path_1)
        name2 = get_cartel_member_name_from_path(img_path_2)

        fig, axs = plt.subplots(1,2, figsize=(10,5))
        axs[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
        axs[0].axis('off')
        axs[0].set_title(name1)
        axs[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
        axs[1].axis('off')
        axs[1].set_title(name2)
        plt.subtitle(f'ArcFace Analysis: {match_status} (similarity: {similarity:.4f})', color=color, fontweight='bold')
        plt.show()
    elif img1 is None or img2 is None:
        print('arcface recognition skipped due to image loading errors.')
    else:
        print('ArcFace recognition skipped due to no faces detected in one or both images.')

test_arcface_recognition('/content/drive/Mydrive/Project_Janus/SinaloaTest/cartelTest3.png',
                         '/content/drive/Mydrive/Project_Janus/Sinaloa/El_Chappo/El_Chappo/clear.png')